# Ensemble Learning to forecast weather

Ensemble has 2 parts: the weak learners and the method of combining these weak learners

There are also 3 types of ensemble learning:
- **Bagging**: each weak learner makes a decision, and the final decision from this ensemble is usually average of decisions or majority vote
- **Boosting**: each weak learner is trained sequentially, and each new model tries to correct the error of the previous one. By focusing on harder and harder to classify problems one weak learner after another, the weighted predictions makes the ensemble much stronger at predicting.
- **Stacking**: Each weak learner is independent, and then a meta-model (a higher level model) learns how to combine these predictions to make the final output. Multi-layer stacking is also possible: feeding first level of base models outputs into a second level and then into a meta-model to create a final output. This can capture a wide variety of patterns in the dataset.

I would try random forest, but I find it hard for forecasting weather as it deals with time.

For this model, I want to use **XGBoost** due to its gradient boosting and its overfitting prevention. Also because it is the most interesting model to me.

### Data Collection

I will try to pull data from the Data.gov.hk website and see if I can get data such as how much rainfall, the highest and lowest temperature, humidity, highest wind speed, the visibility, and the amount of sunshine for each day and try to get like, idk, 5 years worth of data?

Website used:
- Daily Mean Pressure: https://data.gov.hk/en-data/dataset/hk-hko-rss-daily-mean-pressure
- Daily Total Rainfall: https://data.gov.hk/en-data/dataset/hk-hko-rss-daily-total-rainfall
- Daily Max, Mean, Min Temperatures: https://data.gov.hk/en-data/dataset/hk-hko-rss-daily-temperature-info-hko
- Daily Mean Relative Humidity: https://data.gov.hk/en-data/dataset/hk-hko-rss-daily-mean-relative-humidity
- Daily Mean Amount of Clouds: https://data.gov.hk/en-data/dataset/hk-hko-rss-daily-mean-amount-of-cloud

The most recent date in each file is 5/31/2025.

A total of 7 csv files (Temperature has 3). Let's dig in!

### Libraries

In [244]:
import pandas as pd
import numpy as np

import xgboost as xgb

from sklearn.model_selection import train_test_split
from sklearn.metrics import log_loss, accuracy_score
from sklearn.metrics import confusion_matrix, classification_report

import plotly.express as px
import plotly.graph_objects as go

### Data Cleaning/Wrangling
A few things to clarify so you don't have to open the CSVs yourself:

**At the top**, it has this message:
- 日平均雲量(百分比) - 天文台
- Daily Mean Amount of Cloud (%) at the Hong Kong Observatory
- 年/Year,月/Month,日/Day,數值/Value,數據完整性/data Completeness

**At the bottom**, it has this message:
- \- "*** 沒有數據/unavailable"
- "# 數據不完整/data incomplete"
- "C 數據完整/data Complete"

So I need to get rid of these and adjust accordingly, too.

Sorry this markdown chunk looks really disgusting, I am still not the best with markdown right now I need to learn that too.

In [245]:
def clean_data(dat, rename_str):
    dat.columns = ['Year', 'Month', 'Day', 'Value', 'Data Complete']
    dat = dat[:-3]
    # print("-----Before cleaning-----")

    # datDefects = len(dat[dat['Data Complete'] != 'C'])
    # print("Number of faulty rows:",datDefects)
    # print("\n---Summary of Value---")
    # print(dat['Value'].describe())
    # print("\n---Summary of Data Complete---")
    # print(dat['Data Complete'].describe())
    
    # Missing data
    dat = dat[dat['Data Complete'] == 'C']

    dat['Year'] = dat['Year'].astype(int).astype(str)
    dat['Month'] = dat['Month'].astype(int).astype(str)
    dat['Day'] = dat['Day'].astype(int).astype(str)

    dat['Date'] = pd.to_datetime(dat['Year']+"-"+dat['Month']+"-"+dat['Day'], format="%Y-%m-%d")
    dat = dat.set_index('Date')
    
    dat = dat.drop(['Year', 'Month', 'Day', 'Data Complete'], axis=1)

    dat = dat.loc[:, ['Value']]
    dat.loc[dat['Value'] == 'Trace', 'Value'] = 0
    dat['Value'] = dat['Value'].astype(float)

    dat = dat.rename(columns = {'Value': rename_str})

    print("-----After cleaning-----")
    print(f"Number of faulty rows: {dat.isna().sum().sum()}")
    # print("---Summary of Data Complete---")
    # print(dat['Data Complete'].describe())
    print("---Dataframe Info---")
    print(dat.info())
    return dat

Cleaning HKO_cloud_amount.csv

In [246]:
cloud = pd.read_csv("data/HKO_cloud_amount.csv", header=2)
cloud = clean_data(cloud, 'Cloud')
cloud.head()

-----After cleaning-----
Number of faulty rows: 0
---Dataframe Info---
<class 'pandas.core.frame.DataFrame'>
DatetimeIndex: 27910 entries, 1949-01-01 to 2025-05-31
Data columns (total 1 columns):
 #   Column  Non-Null Count  Dtype  
---  ------  --------------  -----  
 0   Cloud   27910 non-null  float64
dtypes: float64(1)
memory usage: 436.1 KB
None


,Cloud
Date,
1949-01-01,68.0
1949-01-02,92.0
1949-01-03,90.0
1949-01-04,94.0
1949-01-05,97.0


Cleaning HKO_max_temp.csv

In [247]:
maxTemp = pd.read_csv("data/HKO_max_temp.csv", header=2)
maxTemp = clean_data(maxTemp, 'MaxTemp')
maxTemp.head()

-----After cleaning-----
Number of faulty rows: 0
---Dataframe Info---
<class 'pandas.core.frame.DataFrame'>
DatetimeIndex: 49094 entries, 1884-01-01 to 2025-05-31
Data columns (total 1 columns):
 #   Column   Non-Null Count  Dtype  
---  ------   --------------  -----  
 0   MaxTemp  49094 non-null  float64
dtypes: float64(1)
memory usage: 767.1 KB
None


,MaxTemp
Date,
1884-01-01,15.3
1884-01-02,17.1
1884-01-03,19.6
1884-01-04,23.2
1884-01-05,19.4


Cleaning HKO_mean_temp.csv

In [248]:
meanTemp = pd.read_csv("data/HKO_mean_temp.csv", header=2)
meanTemp = clean_data(meanTemp, 'MeanTemp')
meanTemp.head()

-----After cleaning-----
Number of faulty rows: 0
---Dataframe Info---
<class 'pandas.core.frame.DataFrame'>
DatetimeIndex: 49003 entries, 1884-04-01 to 2025-05-31
Data columns (total 1 columns):
 #   Column    Non-Null Count  Dtype  
---  ------    --------------  -----  
 0   MeanTemp  49003 non-null  float64
dtypes: float64(1)
memory usage: 765.7 KB
None


,MeanTemp
Date,
1884-04-01,17.8
1884-04-02,14.6
1884-04-03,14.8
1884-04-04,16.4
1884-04-05,18.3


Cleaning HKO_min_temp.csv

In [249]:
minTemp = pd.read_csv("data/HKO_min_temp.csv", header=2)
minTemp = clean_data(minTemp, 'MinTemp')
minTemp.head()

-----After cleaning-----
Number of faulty rows: 0
---Dataframe Info---
<class 'pandas.core.frame.DataFrame'>
DatetimeIndex: 49094 entries, 1884-01-01 to 2025-05-31
Data columns (total 1 columns):
 #   Column   Non-Null Count  Dtype  
---  ------   --------------  -----  
 0   MinTemp  49094 non-null  float64
dtypes: float64(1)
memory usage: 767.1 KB
None


,MinTemp
Date,
1884-01-01,13.7
1884-01-02,14.6
1884-01-03,16.2
1884-01-04,17.0
1884-01-05,13.6


Cleaning HKO_pressure.csv

In [250]:
pressure = pd.read_csv("data/HKO_pressure.csv", header=2)
pressure = clean_data(pressure, 'Pressure')
pressure.head()

-----After cleaning-----
Number of faulty rows: 0
---Dataframe Info---
<class 'pandas.core.frame.DataFrame'>
DatetimeIndex: 49003 entries, 1884-04-01 to 2025-05-31
Data columns (total 1 columns):
 #   Column    Non-Null Count  Dtype  
---  ------    --------------  -----  
 0   Pressure  49003 non-null  float64
dtypes: float64(1)
memory usage: 765.7 KB
None


,Pressure
Date,
1884-04-01,1014.9
1884-04-02,1018.0
1884-04-03,1014.8
1884-04-04,1011.2
1884-04-05,1011.6


Cleaning HKO_rainfall.csv

In [251]:
rainfall = pd.read_csv("data/HKO_rainfall.csv", header=2)
rainfall = clean_data(rainfall, 'Rainfall')
rainfall.head()

-----After cleaning-----
Number of faulty rows: 0
---Dataframe Info---
<class 'pandas.core.frame.DataFrame'>
DatetimeIndex: 49034 entries, 1884-03-01 to 2025-05-31
Data columns (total 1 columns):
 #   Column    Non-Null Count  Dtype  
---  ------    --------------  -----  
 0   Rainfall  49034 non-null  float64
dtypes: float64(1)
memory usage: 766.2 KB
None


,Rainfall
Date,
1884-03-01,0.0
1884-03-02,0.0
1884-03-03,0.0
1884-03-04,0.0
1884-03-05,0.0


Cleaning HKO_relative_humidity.csv

In [252]:
humidity = pd.read_csv("data/HKO_relative_humidity.csv", header=2)
humidity = clean_data(humidity, 'Humidity')
humidity.head()

-----After cleaning-----
Number of faulty rows: 0
---Dataframe Info---
<class 'pandas.core.frame.DataFrame'>
DatetimeIndex: 28640 entries, 1947-01-01 to 2025-05-31
Data columns (total 1 columns):
 #   Column    Non-Null Count  Dtype  
---  ------    --------------  -----  
 0   Humidity  28640 non-null  float64
dtypes: float64(1)
memory usage: 447.5 KB
None


,Humidity
Date,
1947-01-01,85.0
1947-01-02,86.0
1947-01-03,84.0
1947-01-04,85.0
1947-01-05,85.0


Now we need to combine all of these dataframes into 1 dataframe.

In [253]:
datas = [cloud, maxTemp, meanTemp, minTemp, pressure, rainfall, humidity]

df = pd.merge(datas[0], datas[1], on='Date', how='inner')

for i in range(2, len(datas)):
    df = pd.merge(df, datas[i], on='Date', how='inner')

df['Raining'] = df['Rainfall'] > 0

df = df.reset_index()

print(df.shape)
print(df.head())

(27909, 9)
        Date  Cloud  MaxTemp  MeanTemp  MinTemp  Pressure  Rainfall  Humidity  \
0 1949-01-01   68.0     19.8      17.3     14.7    1013.8       0.0      81.0   
1 1949-01-02   92.0     19.5      18.3     17.6    1014.8       0.2      89.0   
2 1949-01-03   90.0     19.0      15.7     12.6    1018.0       1.2      85.0   
3 1949-01-04   94.0     15.1      13.3     10.7    1020.8       0.0      75.0   
4 1949-01-05   97.0     15.2      12.3      9.7    1022.2       0.0      72.0   

   Raining  
0    False  
1     True  
2     True  
3    False  
4    False  


# Data features and Data Splitting
Finally after gathering and wrangling and cleaning all that data, let's start getting it ready for the model!

- Features: We want to predict if it will rain or not, so the features will be everything except Rainfall and Raining.
- Target: If it is raining or not, which we have prepared as the `Raining` column.

In [ ]:
features = ['Cloud', 'MaxTemp', 'MeanTemp', 'MinTemp', 'Pressure', 'Humidity']
X = df[features]
y = df['Raining'].astype(int)

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.4, shuffle=False)

print(X_train.shape, y_train.shape)
print(X_test.shape, y_test.shape)

(16745, 6) (16745,)
(11164, 6) (11164,)


# XGBoost Model and logloss graph
Initialize using the XGBClassifier library (hehe simple isn't it)

In [347]:
model = xgb.XGBClassifier(use_label_encoder=False, eval_metric='logloss', random_state=42, n_estimators=100, learning_rate=5*1e-2, max_depth=8)

eval_set = [(X_train, y_train), (X_test, y_test)]
model.fit(X_train, y_train, eval_set=eval_set, verbose=False) # you can set verbose=True if you want to see the logloss for each tree

results = model.evals_result()
epochs = len(results['validation_0']['logloss'])
x_axis = list(range(0, epochs))

fig = go.Figure()
fig.add_trace(go.Scatter(
    x=x_axis,
    y=results['validation_0']['logloss'],
    mode='lines+markers',
    name='Train'),
)

fig.add_trace(go.Scatter(
    x=x_axis,
    y=results['validation_1']['logloss'],
    mode='lines+markers',
    name='Test'),
)

fig.update_layout(
    title='XGBoost Log Loss',
    xaxis_title='Number of Trees',
    yaxis=dict(
        title='Log Loss',
        tick0 = 0,
        dtick = 0.1,
        # range=[min(results['validation_1']['logloss'])-0.2, max(results['validation_1']['logloss'])+0.2]
    ),
)

fig.show()

### Model evaluation
Here we will show its accuracy, confusion matrix, and its classification report from sklearn

In [348]:
y_pred = model.predict(X_test)
y_pred_proba = model.predict_proba(X_test)[:, 1]

test_accuracy = accuracy_score(y_test, y_pred)

print(f"Accuracy: {100*accuracy_score(y_test, y_pred):.4f}%")
print(confusion_matrix(y_test, y_pred))
print(classification_report(y_test, y_pred))


Accuracy: 82.5063%
[[6308  595]
 [1358 2903]]
              precision    recall  f1-score   support

           0       0.82      0.91      0.87      6903
           1       0.83      0.68      0.75      4261

    accuracy                           0.83     11164
   macro avg       0.83      0.80      0.81     11164
weighted avg       0.83      0.83      0.82     11164



### Prediction for a custom day

In [351]:
predictDate = '2024-11-13'

predict_row = df[df['Date'] == predictDate]

if not predict_row.empty:
    predict_features = predict_row[features]
    prob_rain = model.predict_proba(predict_features)[0][1]
    # Predicted Rain = 1, Predict NO Rain = 0
    will_Rain = model.predict(predict_features)[0]
    print(f"Prediction for {predictDate}:")
    print(f"Chance of rain: {100*prob_rain:.4f}%")
    print(will_Rain)
else:
    print(f"No data available for {predictDate} to make a prediction.")

Prediction for 2024-11-13:
Chance of rain: 36.9028%
0


### Prediction for a range of days

There are 2 graphs here:
- The first graph is the percent chance of rain every day for the set range of days.
- The second graph is the actual amount of rain every day for the set range of days.

In [352]:
date_range = pd.date_range(start='2024-11-01', end='2025-01-01', freq='D') # inclusive

rain_chances = []
dates_with_data = []

for date in date_range:
    # same as above for a single day, copy paste
    date_str = date.strftime('%Y-%m-%d')
    row = df[df['Date'] == date_str]

    if not row.empty:
        features_row = row[features]
        prob = model.predict_proba(features_row)[0][1]
        rain_chances.append(prob * 100)
        dates_with_data.append(date)
    else:
        rain_chances.append(None)
        dates_with_data.append(date)

fig = go.Figure()

fig.add_shape(
    type="line",
    x0=dates_with_data[0], y0=50,
    x1=dates_with_data[-1], y1=50,
    line=dict(color="red", dash="dot"),
)
fig.add_trace(go.Scatter(
    x=dates_with_data,
    y=rain_chances,
    mode='lines+markers',
    name='Chance of Rain (%)'
))

fig.update_layout(
    title='Predicted Chance of Rain between 2024-11-01 and 2025-01-01',
    xaxis=dict(
        title='Date',
        tickformat='%Y-%m-%d',
        tickmode='array',
        tickvals=dates_with_data,
        ticktext=[d.strftime('%Y-%m-%d') for d in dates_with_data],
        tickangle=30
    ),
    yaxis=dict(
        title='Percent Chance of Rain',
        tickvals=list(range(0, 101, 10)),
        range=[0, 100]
    ),
)
fig.show()


In [358]:
rainfall_amounts = []
rainfall_dates = []

for date in date_range:
    # copy pasting again lol
    date_str = date.strftime('%Y-%m-%d')
    row = df[df['Date'] == date_str]
    if not row.empty:
        rainfall = row['Rainfall'].values[0]
        rainfall_amounts.append(rainfall)
        rainfall_dates.append(date)
    else:
        rainfall_amounts.append(None)
        rainfall_dates.append(date)

fig_rainfall = go.Figure()

fig_rainfall.add_trace(go.Bar(
    x=rainfall_dates,
    y=rainfall_amounts,
    # mode='lines+markers',
    name='Rainfall (mm)'
))

fig_rainfall.update_layout(
    title='Daily Rainfall Amount',
    xaxis=dict(
        title='Date',
        tickformat='%Y-%m-%d',
        tickmode='array',
        tickvals=rainfall_dates,
        ticktext=[d.strftime('%Y-%m-%d') for d in rainfall_dates],
        tickangle=30,
    ),
    yaxis=dict(
        title='Rainfall (mm)',
        tickvals=list(range(0, int(max(rainfall_amounts))+10, 5)),
        range=[0, int(max(rainfall_amounts))+10]
    ),
)

fig_rainfall.show()
